In [4]:
import ROOT
import math

# =========================
# USER SETTINGS
# =========================
root_file = "tumor.root"
tree_name = "t"

detector_vlm = 4          # change to your detector volume ID
source_energy = 662.0     # keV, change e.g. 113.0 for Lu-177

fit_min = 600.0           # keV, for 662 keV peak
fit_max = 720.0           # keV

# =========================
# OPEN FILE
# =========================
f = ROOT.TFile.Open(root_file)
if not f or f.IsZombie():
    raise RuntimeError("Cannot open ROOT file")

t = f.Get(tree_name)
if not t:
    raise RuntimeError("Cannot find tree")

# =========================
# HISTOGRAM
# =========================
h = ROOT.TH1F("h", "Detector energy spectrum;Energy deposited (keV);Counts",
              300, 0, source_energy * 1.2)

# Fill energy deposition in detector
# If de is per step, this plots step energy deposits, not event total.
for event in t:
    n = len(event.vlm)

    total_edep = 0.0

    for i in range(n):
        if event.vlm[i] == detector_vlm:
            total_edep += event.de[i]

    if total_edep > 0:
        h.Fill(total_edep)

# =========================
# FIT PHOTOPEAK
# =========================
fit = ROOT.TF1("fit", "gaus", fit_min, fit_max)
h.Fit(fit, "R")

mean = fit.GetParameter(1)
sigma = fit.GetParameter(2)

fwhm = 2.355 * sigma
resolution = (100.0*fwhm)/mean

print("Peak mean =", mean, "keV")
print("Sigma =", sigma, "keV")
print("FWHM =", fwhm, "keV")
print("Energy resolution =", resolution, "%")

# =========================
# DRAW
# =========================
c = ROOT.TCanvas("c", "Energy Resolution", 900, 700)
h.Draw()
fit.Draw("same")

text = ROOT.TLatex()
text.SetNDC()
text.SetTextSize(0.035)
text.DrawLatex(0.55, 0.80, f"Mean = {mean:.2f} keV")
text.DrawLatex(0.55, 0.75, f"Sigma = {sigma:.2f} keV")
text.DrawLatex(0.55, 0.70, f"FWHM = {fwhm:.2f} keV")
text.DrawLatex(0.55, 0.65, f"Resolution = {resolution:.2f}%")

c.Draw()

Peak mean = 670.2102151994535 keV
Sigma = 2.3223679400605923 keV
FWHM = 5.469176498842695 keV
Energy resolution = 0.8160389643143647 %
****************************************
         Invalid FitResult  (status = 4 )
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      47.5589
NDf                       =           19
Edm                       =  0.000397417
NCalls                    =         1753
Constant                  =       217102   +/-   95322.5     
Mean                      =       670.21   +/-   0.612605    
Sigma                     =      2.32237   +/-   0.105067     	 (limited)


Warning in <Fit>: Abnormal termination of minimization.
Info in <TCanvas::MakeDefCanvas>:  created default TCanvas with name c1
